# Notebook 4: Simulation-Based Inference from Scratch

In this notebook, we will introduce **Simulation-Based Inference (SBI)**, also known as Likelihood-Free Inference (LFI), and implement its two most common methods ourselves, using nothing but PyTorch and the conditional flow from Notebook 3.

**What you will learn**
- what problem SBI solves, and which ingredients it needs
- how Neural Posterior Estimation (NPE) works: a conditional flow and about 20 lines of training code
- how Neural Likelihood Estimation (NLE) works, including a simple MCMC sampler
- how a coverage test (TARP) checks whether the posteriors can be trusted

**Runtime:** about one minute on a laptop.

The notebooks which follow solve the same problem with ready-made packages (`sbi`, `ltu-ili`, `sbikit`). After this notebook you will know what they do under the hood.

## Concept

In many areas of science (like astrophysics), we have a complex simulation (forward model) that predicts data $x$ given parameters $\theta$. However, the likelihood $p(x|\theta)$ is often intractable or too expensive to compute, so we cannot use the usual tools (like MCMC with an analytic likelihood) to obtain the posterior

$$ p(\theta|x) = \frac{p(x|\theta)\, p(\theta)}{p(x)} $$

SBI only needs two ingredients:
1. a **prior** $p(\theta)$ we can sample from,
2. a **simulator** which produces data $x$ for given parameters $\theta$, i.e. which draws samples from $p(x|\theta)$.

With these we generate pairs $(\theta_i, x_i)$ and train a conditional density estimator on them, exactly like the conditional flow at the end of Notebook 3:
- **NPE (Neural Posterior Estimation)** learns the posterior $p(\theta|x)$ directly: the parameters are the target, the data are the context.
- **NLE (Neural Likelihood Estimation)** learns the likelihood $p(x|\theta)$: the data are the target, the parameters are the context. The posterior then follows from Bayes' theorem and MCMC.

In [ ]:
import time
import inspect
import numpy as np
import torch
import torch.optim as optim
import matplotlib.pyplot as plt

# The conditional flow from the end of Notebook 3
from flow_utils import ConditionalRealNVP
# The toy problem which we share with the following notebooks
from tutorial_utils import (gaussian_simulator, sample_gaussian_prior, gaussian_benchmark,
                            GAUSSIAN_PRIOR_LOW, GAUSSIAN_PRIOR_HIGH,
                            analytic_posterior, analytic_moments, plot_posterior)

torch.manual_seed(42)
np.random.seed(42)

# Our models are small, so they do not benefit from many CPU threads. Limiting the thread
# count avoids large slow-downs from thread oversubscription (common on laptops).
torch.set_num_threads(4)

## 1. The Problem: a Gaussian Toy Model

We will infer the mean and the (logarithm of the) standard deviation of a Gaussian distribution from 10 samples drawn from it.

- **Parameters:** $\theta = (\mu, \log \sigma)$
- **Prior:** uniform, $\mu \in [-5, 5]$ and $\log \sigma \in [-2, 1]$
- **Simulator:** draws ten numbers from $\mathcal{N}(\mu, \sigma^2)$. This is our data vector $x$.

The simulator lives in the file `tutorial_utils.py`, because the next notebooks use it as well. Here is its source code:

In [ ]:
print(inspect.getsource(gaussian_simulator))

# Run the simulator a few times for the same parameters: the output is random
theta_example = torch.tensor([0.0, 0.0])    # mean = 0, std = 1
for i in range(3):
    print(gaussian_simulator(theta_example).numpy().round(2))

### Training Simulations

We draw parameters from the prior and run the simulator once for each of them. To make the notebooks comparable, we use a fixed set of 10000 training simulations, 500 test simulations and one "observation" $x_{obs}$, which was simulated with $\theta_{true} = (2, -0.5)$. The function `gaussian_benchmark` does nothing but

```python
theta_train = sample_gaussian_prior(10000)
x_train = gaussian_simulator(theta_train)
```

with a fixed random seed.

In [ ]:
benchmark = gaussian_benchmark()
theta_train, x_train = benchmark.theta_train, benchmark.x_train
theta_test, x_test = benchmark.theta_test, benchmark.x_test
theta_true, x_obs = benchmark.theta_obs, benchmark.x_obs

print("Training parameters:", theta_train.shape)
print("Training data:      ", x_train.shape)
print("Observation:        ", x_obs.numpy().round(2))

fig, axs = plt.subplots(1, 2, figsize=(13, 4.5))
axs[0].scatter(theta_train[:2000, 0], theta_train[:2000, 1], s=2)
axs[0].plot(theta_true[0], theta_true[1], '*', color='red', markersize=12, label='True parameters of the observation')
axs[0].set(xlabel='Mean', ylabel='Log Std', title='Training parameters: samples from the prior')
axs[0].legend(loc='upper right')

for i in range(6):
    axs[1].plot(x_train[i], np.full(10, i), 'o', markersize=4)
    axs[1].text(6.5, i, rf'$\mu={theta_train[i, 0]:.1f},\ \log\sigma={theta_train[i, 1]:.1f}$', va='center')
axs[1].set(xlabel='x', ylabel='Simulation', xlim=(-8, 12), title='Six training simulations (ten numbers each)')
plt.show()

### The Exact Posterior (for Comparison)

SBI is made for simulators whose likelihood we cannot write down. Our toy model is deliberately *not* such a case: the likelihood is a product of ten Gaussians, the prior is flat, and with only two parameters we can simply evaluate the posterior on a grid:

$$ \log p(\theta|x) = -N \log \sigma - \sum_{i=1}^{N} \frac{(x_i - \mu)^2}{2 \sigma^2} + \text{const} \qquad \text{with } \theta = (\mu, \log \sigma) $$

This gives us the correct answer to compare with, something we do not have in a real application. The function `analytic_posterior` in `tutorial_utils.py` does this calculation. Note that the true parameters do not have to be at the peak of the posterior: with only 10 data points, the posterior is centred on the mean and standard deviation of this particular noisy sample.

In [ ]:
mean_grid, log_std_grid, posterior_grid = analytic_posterior(x_obs)

plt.figure(figsize=(6, 4.5))
plt.pcolormesh(mean_grid, log_std_grid, posterior_grid.T, cmap='Blues')
plt.plot(theta_true[0], theta_true[1], '*', color='red', markersize=12, label='True parameters')
plt.xlim(1, 3)
plt.ylim(-1.5, 0.5)
plt.xlabel('Mean')
plt.ylabel('Log Std')
plt.title('Exact posterior of the observation')
plt.legend()
plt.show()

## 2. Neural Posterior Estimation (NPE)

NPE trains a conditional flow $q(\theta|x)$ to approximate the posterior directly. The recipe:

1. Draw parameters from the prior, $\theta_i \sim p(\theta)$, and simulate $x_i \sim p(x|\theta_i)$. (Done: these are our training simulations.)
2. Train the flow by minimizing the negative log-likelihood of the parameters given the data:
$$ \mathcal{L} = -\frac{1}{N} \sum_i \log q(\theta_i | x_i) $$
3. For an observation $x_{obs}$, sample from $q(\theta | x_{obs})$.

Why does this give the posterior? The pairs $(\theta_i, x_i)$ are samples from the joint distribution $p(\theta, x) = p(x|\theta)\,p(\theta) = p(\theta|x)\,p(x)$. For every fixed $x$, the parameters which come with it are therefore distributed according to $p(\theta|x)$, and the loss is minimal when $q(\theta|x)$ equals this distribution. Note that the prior enters through the training set: the flow learns the posterior for the prior the parameters were drawn from.

### Normalization

As in Notebook 1, we normalize the inputs and outputs of the network. This matters here: the posterior is much narrower than the prior, and its width changes a lot from one observation to the next.

In [ ]:
# Normalization constants from the training set
theta_mean, theta_std = theta_train.mean(dim=0), theta_train.std(dim=0)
x_mean, x_std = x_train.mean(), x_train.std()      # a single number for all ten (exchangeable) entries

def normalize_theta(theta):
    return (theta - theta_mean) / theta_std

def normalize_x(x):
    return (x - x_mean) / x_std

### Training Loop

The training loop is the one from Notebook 2: mini-batches, Adam, and a validation set. We keep the weights with the best validation loss. Since we need the same loop for NLE below, we write it as a function for a general conditional flow $q(\text{target}|\text{context})$.

In [ ]:
def train_flow(flow, target, context, epochs=150, batch_size=250, lr=2e-3, validation_fraction=0.1):
    # Split into training and validation data
    num_val = int(validation_fraction * len(target))
    target_val, context_val = target[:num_val], context[:num_val]
    target_train, context_train = target[num_val:], context[num_val:]

    optimizer = optim.Adam(flow.parameters(), lr=lr)
    # Reduce the learning rate step by step to zero (helps to converge to a good minimum)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    best_val_loss, best_state = float('inf'), None
    train_losses, val_losses = [], []

    for epoch in range(epochs):
        flow.train()
        permutation = torch.randperm(len(target_train))       # shuffle the training data
        epoch_loss = 0.
        for i in range(0, len(target_train), batch_size):
            batch = permutation[i:i + batch_size]
            optimizer.zero_grad()
            # Negative log-likelihood of the targets given the context
            loss = -flow.log_prob(target_train[batch], context_train[batch]).mean()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * len(batch)
        scheduler.step()
        train_losses.append(epoch_loss / len(target_train))

        flow.eval()
        with torch.no_grad():
            val_loss = -flow.log_prob(target_val, context_val).mean().item()
        val_losses.append(val_loss)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {name: value.clone() for name, value in flow.state_dict().items()}

    flow.load_state_dict(best_state)        # restore the best weights
    flow.eval()
    print(f'Best validation loss: {best_val_loss:.4f}')
    return train_losses, val_losses

In [ ]:
# q(theta | x): the target has 2 dimensions (the parameters), the context has 10 (the data)
# max_log_scale: each layer may rescale by up to exp(+-3), since posteriors are much narrower than the prior
npe_flow = ConditionalRealNVP(input_dim=2, context_dim=10, hidden_dim=64, n_layers=8, max_log_scale=3.0)

start = time.time()
train_losses, val_losses = train_flow(npe_flow, target=normalize_theta(theta_train), context=normalize_x(x_train))
print(f'Training time: {time.time() - start:.0f} s')

plt.plot(train_losses, label='Training')
plt.plot(val_losses, label='Validation')
plt.xlabel('Epoch')
plt.ylabel(r'$-\log q(\theta|x)$')
plt.title('NPE training')
plt.legend()
plt.show()

### Inference on the Observation

Sampling from the posterior is now a single forward pass of the flow in the generative direction: we draw $z \sim \mathcal{N}(0, I)$ and transform it with $x_{obs}$ as context. Finally we undo the normalization of the parameters, and discard the (few) samples outside of the prior range.

In [ ]:
def sample_npe(x, num_samples):
    """Samples from q(theta | x) for a single observation x [1, 10]."""
    context = normalize_x(x).expand(num_samples, -1)       # the same observation for every sample
    with torch.no_grad():
        theta = npe_flow.sample(context) * theta_std + theta_mean
    inside_prior = ((theta >= GAUSSIAN_PRIOR_LOW) & (theta <= GAUSSIAN_PRIOR_HIGH)).all(dim=1)
    return theta[inside_prior]

start = time.time()
samples_npe = sample_npe(x_obs, 5000)
print(f'NPE: {time.time() - start:.3f} s for {len(samples_npe)} posterior samples')

exact_mean, exact_std = analytic_moments(x_obs)
print('Posterior mean, NPE:  ', samples_npe.mean(dim=0).numpy().round(3), ' exact:', exact_mean.numpy().round(3))
print('Posterior std,  NPE:  ', samples_npe.std(dim=0).numpy().round(3), ' exact:', exact_std.numpy().round(3))

plot_posterior({'NPE (from scratch)': samples_npe}, x_obs, theta_true)

The network never saw the likelihood, only simulations, and still gets close to the exact posterior. The agreement is not perfect: with our simple flow and 10000 simulations, the NPE posterior is typically a bit too wide, in particular for $\log \sigma$ (the network has to work out by itself that the scatter of the ten numbers is what matters). Also note what we did *not* have to do: no MCMC, and no new training for this particular observation. The same network gives the posterior for any other observation, which is called **amortized** inference (more on this in the next notebook).

## 3. Neural Likelihood Estimation (NLE)

NLE uses the same training set and the same kind of flow, but swaps the roles: it learns the likelihood $q(x|\theta) \approx p(x|\theta)$, with the data as target and the parameters as context.

$$ \mathcal{L} = -\frac{1}{N} \sum_i \log q(x_i | \theta_i) $$

The flow now replaces the analytic likelihood which we do not have. The posterior follows from Bayes' theorem,

$$ \log p(\theta|x_{obs}) = \log q(x_{obs}|\theta) + \log p(\theta) + \text{const} $$

and, as in a classical analysis, we need MCMC to draw samples from it.

In [ ]:
# q(x | theta): the target has 10 dimensions (the data), the context has 2 (the parameters)
nle_flow = ConditionalRealNVP(input_dim=10, context_dim=2, hidden_dim=64, n_layers=8, max_log_scale=3.0)

start = time.time()
# The likelihood of this model is easier to learn than the posterior, so fewer epochs are enough
train_losses, val_losses = train_flow(nle_flow, target=normalize_x(x_train), context=normalize_theta(theta_train), epochs=60)
print(f'Training time: {time.time() - start:.0f} s')

plt.plot(train_losses, label='Training')
plt.plot(val_losses, label='Validation')
plt.xlabel('Epoch')
plt.ylabel(r'$-\log q(x|\theta)$')
plt.title('NLE training')
plt.legend()
plt.show()

### The Log-Posterior

The (unnormalized) log-posterior is the learned log-likelihood of the observation plus the log-prior. Our prior is flat, so the log-prior is a constant inside of the prior range and $-\infty$ outside.

In [ ]:
def log_posterior_nle(theta, x):
    """Unnormalized log-posterior for parameters theta [n, 2] and a single observation x [1, 10]."""
    with torch.no_grad():
        log_likelihood = nle_flow.log_prob(normalize_x(x).expand(len(theta), -1), normalize_theta(theta))
    inside_prior = ((theta >= GAUSSIAN_PRIOR_LOW) & (theta <= GAUSSIAN_PRIOR_HIGH)).all(dim=1)
    return torch.where(inside_prior, log_likelihood, torch.full_like(log_likelihood, -float('inf')))

### A Simple MCMC Sampler

We use the simplest MCMC algorithm, random-walk Metropolis:
1. From the current position $\theta$, propose a new one, $\theta' = \theta + \text{step size} \cdot \mathcal{N}(0, 1)$.
2. Accept the proposal with probability $\min\left(1, p(\theta'|x) / p(\theta|x)\right)$, otherwise stay where you are.

After a *burn-in* phase the positions are samples from the posterior. We run many independent walkers at the same time, which costs almost nothing extra: the flow evaluates all of them in one batch.

In [ ]:
def metropolis(log_prob_fn, theta_0, num_steps, step_size):
    """Random-walk Metropolis for many walkers in parallel.

    theta_0: initial positions [num_walkers, num_params]
    Returns the chain [num_steps, num_walkers, num_params] and the acceptance fraction.
    """
    theta = theta_0.clone()
    log_p = log_prob_fn(theta)
    chain = torch.empty((num_steps, *theta.shape))
    num_accepted = 0

    for step in range(num_steps):
        proposal = theta + step_size * torch.randn_like(theta)
        log_p_proposal = log_prob_fn(proposal)

        # Accept with probability min(1, p_proposal / p)
        accept = torch.log(torch.rand(len(theta))) < log_p_proposal - log_p
        theta = torch.where(accept[:, None], proposal, theta)
        log_p = torch.where(accept, log_p_proposal, log_p)

        num_accepted += accept.sum().item()
        chain[step] = theta

    return chain, num_accepted / (num_steps * len(theta))

num_walkers = 100
num_steps = 1500
burn_in = 500

start = time.time()
theta_0 = sample_gaussian_prior(num_walkers)                # start the walkers at random positions in the prior
chain, acceptance_fraction = metropolis(lambda theta: log_posterior_nle(theta, x_obs), theta_0,
                                        num_steps=num_steps, step_size=torch.tensor([0.2, 0.25]))
print(f'NLE: {time.time() - start:.1f} s for the MCMC, acceptance fraction: {acceptance_fraction:.2f}')

# Remove the burn-in, keep every 10th step (successive steps are correlated), and merge the walkers
samples_nle = chain[burn_in::10].reshape(-1, 2)
print('Posterior mean, NLE:  ', samples_nle.mean(dim=0).numpy().round(3), ' exact:', exact_mean.numpy().round(3))
print('Posterior std,  NLE:  ', samples_nle.std(dim=0).numpy().round(3), ' exact:', exact_std.numpy().round(3))

# The walkers start all over the prior and find the posterior during the burn-in
fig, axs = plt.subplots(1, 2, figsize=(13, 3.5))
for i, name in enumerate(['Mean', 'Log Std']):
    axs[i].plot(chain[:, :20, i], lw=0.5)
    axs[i].axvline(burn_in, color='black', ls='--', label='End of burn-in')
    axs[i].set(xlabel='Step', ylabel=name, title=f'Chains of 20 walkers: {name}')
axs[0].legend()
plt.show()

plot_posterior({'NPE (from scratch)': samples_npe, 'NLE (from scratch)': samples_nle}, x_obs, theta_true)

NLE is almost indistinguishable from the exact posterior here: the likelihood of this model (ten independent Gaussian numbers) is easy for a flow to learn, easier than the posterior. This is not a general rule. Which of the two methods works better depends on the problem. Their trade-offs:

| | NPE | NLE |
|---|---|---|
| learns | posterior $p(\theta\vert x)$ | likelihood $p(x\vert\theta)$ |
| flow dimension | number of parameters | size of the data vector |
| posterior samples | one forward pass (amortized) | MCMC for every observation |
| prior | fixed by the training set | free to choose when sampling |

## 4. Can we Trust the Posteriors? A Coverage Test from Scratch

One observation with a known answer is not a validation. In a real application there is no exact posterior to compare with, but we can always test a necessary property of correct posteriors: **calibration**. If a region contains $x\%$ of the posterior probability mass, it should contain the true parameters in $x\%$ of the cases.

We can check this with simulations, because for a simulation we know the true parameters. The **TARP** method (Tests of Accuracy with Random Points, [Lemos et al., 2023](https://arxiv.org/abs/2302.03026)) only requires samples from the estimated posterior:

1. Take test simulations $(\theta_{test}, x_{test})$ with $\theta_{test}$ drawn from the prior, and sample from the estimated posterior $q(\theta|x_{test})$ of each.
2. For each test simulation, draw a random reference point $\theta_{ref}$ in parameter space and compute the fraction of posterior samples that are closer to $\theta_{ref}$ than the true $\theta_{test}$. This is the credibility level of the smallest ball around $\theta_{ref}$ that contains the true parameters.
3. If the posteriors are correct, these fractions are uniformly distributed. We therefore plot their cumulative distribution (the expected coverage as a function of the credibility level), which should follow the diagonal.

For NPE, sampling the posteriors of all 500 test simulations takes a fraction of a second. (For NLE we would need 500 MCMC runs. We come back to this in Notebook 7.)

In [ ]:
# 1. Posterior samples for all test simulations at once: [num_samples, num_test, num_params]
num_samples = 1000
num_test = len(x_test)
context = normalize_x(x_test).repeat(num_samples, 1)            # every test observation num_samples times
with torch.no_grad():
    posterior_samples = npe_flow.sample(context) * theta_std + theta_mean
posterior_samples = posterior_samples.reshape(num_samples, num_test, 2)

def tarp_from_scratch(posterior_samples, theta_true, num_bins=20):
    # Rescale the parameters to the unit square, so that both parameters count equally in the distances
    samples = (posterior_samples - GAUSSIAN_PRIOR_LOW) / (GAUSSIAN_PRIOR_HIGH - GAUSSIAN_PRIOR_LOW)
    truth = (theta_true - GAUSSIAN_PRIOR_LOW) / (GAUSSIAN_PRIOR_HIGH - GAUSSIAN_PRIOR_LOW)

    # 2. One random reference point per test simulation
    reference = torch.rand_like(truth)
    distance_truth = (truth - reference).norm(dim=-1)              # [num_test]
    distance_samples = (samples - reference).norm(dim=-1)          # [num_samples, num_test]
    # Fraction of posterior samples which are closer to the reference point than the truth
    credibility = (distance_samples < distance_truth).float().mean(dim=0)

    # 3. Cumulative distribution of these fractions
    alpha = torch.linspace(0, 1, num_bins + 1)
    expected_coverage = (credibility[None, :] <= alpha[:, None]).float().mean(dim=1)
    return alpha, expected_coverage

alpha, ecp = tarp_from_scratch(posterior_samples, theta_test)

# The same test with the tarp package
from tarp import get_tarp_coverage
ecp_package, alpha_package = get_tarp_coverage(posterior_samples.numpy(), theta_test.numpy(),
                                               references='random', metric='euclidean', norm=True)

plt.figure(figsize=(6, 6))
plt.plot([0, 1], [0, 1], 'k--', label='Ideal')
plt.plot(alpha, ecp, 'o-', label='NPE, our implementation')
plt.plot(alpha_package, ecp_package, label='NPE, tarp package')
plt.xlabel('Credibility Level')
plt.ylabel('Expected Coverage')
plt.title('TARP Coverage Test')
plt.legend()
plt.show()

The curve stays close to the diagonal: our NPE posteriors are reasonably well calibrated. (The two curves differ slightly because they use different random reference points.) How a failed coverage test looks like is shown in the next notebook.

## Summary

- SBI needs only a simulator and a prior. We simulate pairs $(\theta, x)$ and train a conditional density estimator on them.
- **NPE** trains a flow $q(\theta|x)$ with the loss $-\log q(\theta_i|x_i)$. Posterior samples then cost one forward pass, for any observation.
- **NLE** trains a flow $q(x|\theta)$ with the loss $-\log q(x_i|\theta_i)$ and needs MCMC for every observation. In return, the prior only enters at sampling time.
- A coverage test with simulations tells us whether the posteriors are calibrated, without knowing the exact posterior.

Everything in this notebook fits into about 100 lines of code. What it leaves out is what makes a method robust in practice: better flows, automatic normalization, early stopping, efficient samplers, diagnostics.

**Next:** Notebook 5 solves the same problem with the `sbi` package, which provides all of this.